# FWA-Only Ablation — NSL-KDD

Ablation study: Feature-Weighted Attention (**FWA**) extractor with **no MCL** and **no custom BiLSTM cell**.

Compared to the full MCL-FWA-BiLSTM pipeline, this notebook replaces:
- ❌ **MCL** (CNN prediction-error-filter) → ✅ **Standard `nn.BiLSTM`**
- ❌ **CustomBiLSTM** (MCL-fused RNN cell) → the same `nn.BiLSTM`
- ✅ **FWA** (Feature-Weighted Attention) is kept as-is

The FWA query (`wc`) is replaced by a linear projection of the BiLSTM's final hidden state.

**Pipeline:**
1. Architecture: Standard BiLSTM → FWA → linear head
2. Data: NSL-KDD (120 features → 10×12 spatial input)
3. Phase 1: Train FWA extractor with Cross-Entropy
4. Phase 2: RF & XGBoost on extracted features
5. Results

## 1. Imports

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from xgboost import XGBClassifier

print(f"PyTorch: {torch.__version__}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

## 2. FWA Layer

Paper Algorithm 2, Eqs 1–2. Unchanged from the full model.

```
score_t = v^T · tanh(W1·h_t + W2·wc + b)
FA_t    = softmax_t(score_t)
FAT     = Σ_t FA_t · h_t
Fe      = concat(FBL, FAT)
```

In [ ]:
class FWA(nn.Module):
    def __init__(self, hidden_dim: int, wc_dim: int, attn_dim: int | None = None):
        super().__init__()
        if attn_dim is None:
            attn_dim = hidden_dim
        self.W1 = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.W2 = nn.Linear(wc_dim, attn_dim, bias=True)
        self.v  = nn.Linear(attn_dim, 1, bias=False)

    def forward(
        self,
        h:   torch.Tensor,   # (N, T, 2H) — BiLSTM hidden states
        wc:  torch.Tensor,   # (N, wc_dim) — attention query
        fbl: torch.Tensor,   # (N, 2H)    — final BiLSTM state
    ) -> tuple[torch.Tensor, torch.Tensor]:
        q      = self.W2(wc).unsqueeze(1)                   # (N, 1, attn_dim)
        k      = self.W1(h)                                 # (N, T, attn_dim)
        scores = self.v(torch.tanh(k + q)).squeeze(-1)      # (N, T)
        fa     = torch.softmax(scores, dim=-1)              # (N, T)
        fat    = torch.bmm(fa.unsqueeze(1), h).squeeze(1)   # (N, 2H)
        fe     = torch.cat([fbl, fat], dim=-1)              # (N, 4H)
        return fe, fa

## 3. FWA Extractor

Replaces the MCL+CustomBiLSTM stack with a standard `nn.LSTM(bidirectional=True)`.  
The FWA query `wc` is derived from a linear projection of the BiLSTM final state `fbl`.

In [ ]:
class FWAExtractor(nn.Module):
    """
    FWA-only feature extractor (ablation — no MCL, no custom BiLSTM cell).

    Architecture:
        x (N,10,12)
          └─> nn.LSTM(bidirectional=True)  → h (N,T,2H), fbl (N,2H)
          └─> wc_proj(fbl)                 → wc (N, wc_dim)   [replaces MCL Wc]
          └─> FWA(h, wc, fbl)             → fe (N, 4H)
    """
    def __init__(
        self,
        input_size:  int = 12,      # features per timestep (cols of 10×12 grid)
        hidden_size: int = 64,      # BiLSTM hidden units per direction
        wc_dim:      int = 56,      # attention query dimension (matches MCL output)
        attn_dim:    int | None = None,
        n_lstm_layers: int = 1,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.hidden_size = hidden_size

        # Standard bidirectional LSTM — no custom MCL fusion
        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=n_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_lstm_layers > 1 else 0.0,
        )

        # Linear projection of final BiLSTM state → wc query for FWA
        self.wc_proj = nn.Linear(2 * hidden_size, wc_dim)

        # FWA attention over all hidden states
        self.fwa = FWA(hidden_dim=2 * hidden_size, wc_dim=wc_dim, attn_dim=attn_dim)

        self.feature_dim = 4 * hidden_size

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, dict]:
        # x: (N, 10, 12)
        h_seq, (h_n, _) = self.bilstm(x)          # h_seq: (N, T, 2H)

        # Concatenate final forward and backward hidden states → (N, 2H)
        fbl = torch.cat([h_n[-2], h_n[-1]], dim=-1)

        # Derive attention query from final state (replaces MCL Wc)
        wc = torch.tanh(self.wc_proj(fbl))         # (N, wc_dim)

        fe, fa = self.fwa(h_seq, wc, fbl)          # (N, 4H), (N, T)
        return fe, {"wc": wc, "fbl": fbl, "fa": fa}


class FWAExtractorWithHead(nn.Module):
    """Phase-1 wrapper: FWAExtractor + temporary linear classifier head."""
    def __init__(self, extractor: FWAExtractor, n_classes: int):
        super().__init__()
        self.extractor = extractor
        self.head = nn.Linear(extractor.feature_dim, n_classes)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        fe, _ = self.extractor(x)
        return self.head(fe), fe

    # No MCL constraint needed — kept as no-op for API compatibility
    def apply_mcl_constraint(self) -> None:
        pass

## 4. Training & Extraction Utilities

In [ ]:
@dataclass
class TrainConfig:
    epochs:                  int   = 100
    batch_size:              int   = 256
    lr:                      float = 1e-3
    weight_decay:            float = 0.0
    device:                  str   = "cuda" if torch.cuda.is_available() else "cpu"
    log_every:               int   = 0
    seed:                    int   = 0
    target_val_acc:          float = 0.90
    early_stopping_patience: int   = 10


def _make_loader(X, y, cfg, shuffle):
    ds = TensorDataset(X, y.long())
    return DataLoader(ds, batch_size=cfg.batch_size, shuffle=shuffle,
                      num_workers=0, pin_memory=cfg.device.startswith("cuda"))


@torch.no_grad()
def _eval_loop(model, loader, loss_fn, device):
    model.eval()
    running, seen, correct = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        logits, _ = model(xb)
        loss = loss_fn(logits, yb)
        bs = yb.size(0)
        running += loss.item() * bs; seen += bs
        correct += (logits.argmax(dim=-1) == yb).sum().item()
    return {"val_loss": running / seen, "val_acc": correct / seen}


def train_fwa_extractor(
    X_train, y_train, cfg=None, X_val=None, y_val=None,
    hidden_size=64, wc_dim=56, n_lstm_layers=1,
):
    cfg = cfg or TrainConfig()
    torch.manual_seed(cfg.seed)
    n_classes = int(y_train.max().item()) + 1

    extractor = FWAExtractor(
        input_size=X_train.shape[2],  # 12 (cols of 10x12 grid)
        hidden_size=hidden_size,
        wc_dim=wc_dim,
        n_lstm_layers=n_lstm_layers,
    )
    model = FWAExtractorWithHead(extractor, n_classes=n_classes).to(cfg.device)
    if cfg.device.startswith("cuda") and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    opt     = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.CrossEntropyLoss()

    train_loader = _make_loader(X_train, y_train, cfg, shuffle=True)
    val_loader   = _make_loader(X_val, y_val, cfg, shuffle=False) if X_val is not None else None

    history, best_val_loss, patience_counter = [], float("inf"), 0

    for epoch in range(cfg.epochs):
        model.train()
        running, seen, correct = 0.0, 0, 0
        for step, (xb, yb) in enumerate(train_loader):
            xb, yb = xb.to(cfg.device, non_blocking=True), yb.to(cfg.device, non_blocking=True)
            logits, _ = model(xb)
            loss = loss_fn(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            # No MCL constraint step needed
            bs = yb.size(0)
            running += loss.item() * bs; seen += bs
            correct += (logits.argmax(dim=-1) == yb).sum().item()
            if cfg.log_every and (step + 1) % cfg.log_every == 0:
                print(f"epoch {epoch} step {step+1} loss {running/seen:.4f}")

        row = {"epoch": epoch, "train_loss": running/seen, "train_acc": correct/seen}
        if val_loader:
            vm = _eval_loop(model, val_loader, loss_fn, cfg.device)
            row.update(vm)
        history.append(row)
        if cfg.log_every: print(row)

        if val_loader:
            if vm["val_acc"] >= cfg.target_val_acc:
                print(f"Reached target accuracy {vm['val_acc']:.4f} >= {cfg.target_val_acc}. Stopping.")
                break
            if vm["val_loss"] < best_val_loss:
                best_val_loss = vm["val_loss"]; patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= cfg.early_stopping_patience:
                    print(f"Early stopping after {cfg.early_stopping_patience} non-improving epochs.")
                    break
    return extractor, history


@torch.no_grad()
def extract_features(extractor, X, batch_size=512, device=None):
    device = device or next(extractor.parameters()).device.type
    extractor.eval()
    out = []
    for i in range(0, X.shape[0], batch_size):
        xb = X[i: i + batch_size].to(device, non_blocking=True)
        fe, _ = extractor(xb)
        out.append(fe.cpu())
    return torch.cat(out, dim=0)

## 5. Classifier Utilities

In [ ]:
@dataclass
class TreeConfig:
    n_estimators: int        = 100
    max_depth:    int | None = None
    n_jobs:       int        = -1
    random_state: int        = 0
    extra:        dict       = field(default_factory=dict)


def fit_rf(Fe_train, y_train, cfg=None):
    cfg = cfg or TreeConfig()
    clf = RandomForestClassifier(
        n_estimators=cfg.n_estimators, max_depth=cfg.max_depth,
        n_jobs=cfg.n_jobs, class_weight="balanced",
        random_state=cfg.random_state, **cfg.extra,
    )
    clf.fit(Fe_train, y_train)
    return clf


def fit_xgb(Fe_train, y_train, cfg=None):
    cfg = cfg or TreeConfig()
    clf = XGBClassifier(
        n_estimators=cfg.n_estimators, max_depth=cfg.max_depth,
        n_jobs=cfg.n_jobs, random_state=cfg.random_state,
        eval_metric="mlogloss", **cfg.extra,
    )
    clf.fit(Fe_train, y_train)
    return clf


def evaluate(clf, Fe, y):
    y_pred = clf.predict(Fe)
    avg = "binary" if len(np.unique(y)) == 2 else "macro"
    return {
        "accuracy":         accuracy_score(y, y_pred),
        "precision":        precision_score(y, y_pred, average=avg, zero_division=0),
        "recall":           recall_score(y, y_pred, average=avg, zero_division=0),
        "f1":               f1_score(y, y_pred, average=avg, zero_division=0),
        "confusion_matrix": confusion_matrix(y, y_pred).tolist(),
        "report":           classification_report(y, y_pred, zero_division=0),
    }

## 6. NSL-KDD Data Preprocessing

Identical to the full pipeline: one-hot encodes 3 categorical columns → 120 features → StandardScaler → reshape `(N, 10, 12)`.

In [ ]:
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate",
    "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate",
    "diff_srv_rate", "srv_diff_host_rate", "dst_host_count",
    "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate",
    "dst_host_srv_rerror_rate", "label", "difficulty",
]
CATEGORICAL   = ["protocol_type", "service", "flag"]
DROP_CONSTANT = ["num_outbound_cmds", "is_host_login"]


def load_and_preprocess_nsl(train_path, test_path):
    train_df = pd.read_csv(train_path, header=None, names=COLUMNS)
    test_df  = pd.read_csv(test_path,  header=None, names=COLUMNS)

    for df in (train_df, test_df):
        df.drop(columns=["difficulty"] + DROP_CONSTANT, inplace=True)

    y_bin_tr = (train_df["label"] != "normal").astype(np.int64).to_numpy()
    y_bin_te = (test_df["label"]  != "normal").astype(np.int64).to_numpy()

    train_df.drop(columns="label", inplace=True)
    test_df.drop(columns="label",  inplace=True)

    combined = pd.concat([train_df, test_df], ignore_index=True)
    combined = pd.get_dummies(combined, columns=CATEGORICAL, dtype=np.float32)
    n_train  = len(train_df)
    X_tr = combined.iloc[:n_train].to_numpy(dtype=np.float32)
    X_te = combined.iloc[n_train:].to_numpy(dtype=np.float32)

    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr).astype(np.float32)
    X_te   = scaler.transform(X_te).astype(np.float32)

    assert X_tr.shape[1] == 120, f"Expected 120 features, got {X_tr.shape[1]}"
    X_tr = X_tr.reshape(-1, 10, 12)
    X_te = X_te.reshape(-1, 10, 12)

    return (
        torch.from_numpy(X_tr), torch.from_numpy(y_bin_tr),
        torch.from_numpy(X_te), torch.from_numpy(y_bin_te),
    )

## 7. Paths & Data Loading

In [ ]:
BASE_DIR   = Path(".").resolve().parent
TRAIN_PATH = BASE_DIR / "NSL-KDD" / "KDDTrain+.txt"
TEST_PATH  = BASE_DIR / "NSL-KDD" / "KDDTest+.txt"

print("Loading and preprocessing NSL-KDD...")
X_tr, y_bin_tr, X_te, y_bin_te = load_and_preprocess_nsl(TRAIN_PATH, TEST_PATH)

print(f"Train X shape : {X_tr.shape}")
print(f"Test  X shape : {X_te.shape}")
print(f"Train: Benign={int((y_bin_tr==0).sum())}, Attack={int((y_bin_tr==1).sum())}")
print(f"Test : Benign={int((y_bin_te==0).sum())}, Attack={int((y_bin_te==1).sum())}")

## 8. Hyperparameter Configuration

In [ ]:
# FWA extractor hyperparameters
HIDDEN_SIZE   = 64   # BiLSTM hidden units per direction
WC_DIM        = 56   # attention query projection dimension
N_LSTM_LAYERS = 1    # number of stacked BiLSTM layers

# Training config
cfg = TrainConfig(
    epochs=100,
    batch_size=256,
    lr=1e-3,
    device=DEVICE,
    log_every=10,
    target_val_acc=0.90,
    early_stopping_patience=10,
)

print(f"FWA Extractor — hidden: {HIDDEN_SIZE}, wc_dim: {WC_DIM}, lstm_layers: {N_LSTM_LAYERS}")
print(f"Feature dim  : {4 * HIDDEN_SIZE}")
print(f"Training on  : {cfg.device}")

## 9. Phase 1 — Train FWA Extractor

In [ ]:
print(f"Training FWA extractor for up to {cfg.epochs} epochs...")
extractor, hist = train_fwa_extractor(
    X_tr, y_bin_tr, cfg=cfg,
    X_val=X_te, y_val=y_bin_te,
    hidden_size=HIDDEN_SIZE,
    wc_dim=WC_DIM,
    n_lstm_layers=N_LSTM_LAYERS,
)

## 10. Training History

In [ ]:
hist_df = pd.DataFrame(hist)
print(hist_df[["epoch", "train_loss", "train_acc", "val_loss", "val_acc"]].tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
if "val_loss" in hist_df.columns:
    ax1.plot(hist_df["epoch"], hist_df["val_loss"], label="Val Loss")
ax1.set_title("Loss Curve — FWA Extractor")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()

ax2.plot(hist_df["epoch"], hist_df["train_acc"], label="Train Acc")
if "val_acc" in hist_df.columns:
    ax2.plot(hist_df["epoch"], hist_df["val_acc"], label="Val Acc")
ax2.axhline(0.90, color="red", linestyle="--", alpha=0.5, label="Target 90%")
ax2.set_title("Accuracy Curve — FWA Extractor")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.legend()

plt.tight_layout()
plt.show()

## 11. Feature Extraction

In [ ]:
print("Extracting latent features from trained FWA extractor...")
fe_tr = extract_features(extractor, X_tr, device=cfg.device).numpy()
fe_te = extract_features(extractor, X_te, device=cfg.device).numpy()

print(f"Train features: {fe_tr.shape}")
print(f"Test  features: {fe_te.shape}")

## 12. Phase 2 — Train Random Forest & XGBoost

In [ ]:
tree_cfg = TreeConfig(n_estimators=100)

print("Training Random Forest...")
rf_clf     = fit_rf(fe_tr, y_bin_tr.numpy(), tree_cfg)
rf_metrics = evaluate(rf_clf, fe_te, y_bin_te.numpy())

print("Training XGBoost...")
xgb_clf     = fit_xgb(fe_tr, y_bin_tr.numpy(), tree_cfg)
xgb_metrics = evaluate(xgb_clf, fe_te, y_bin_te.numpy())

## 13. Results

In [ ]:
SKIP = {"confusion_matrix", "report"}
results = pd.DataFrame([
    {"Classifier": "Random Forest", **{k: v for k, v in rf_metrics.items()  if k not in SKIP}},
    {"Classifier": "XGBoost",       **{k: v for k, v in xgb_metrics.items() if k not in SKIP}},
]).set_index("Classifier").round(4)
results.columns = [c.capitalize() for c in results.columns]

print("\n=== FWA-Only Ablation — NSL-KDD ===")
print(results.to_string())
results

In [ ]:
print("\n--- Random Forest Classification Report ---")
print(rf_metrics["report"])
print("\n--- XGBoost Classification Report ---")
print(xgb_metrics["report"])